<a href="https://colab.research.google.com/github/loukikjoshi06-ops/SIH-26016/blob/main/01_qwen2_5_vl_7b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi


Sat Sep 26 04:02:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes peft qwen-vl-utils trl torchvision


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 71.2 MB/s eta 0:00:00


In [ ]:
!pip install -q transformers bitsandbytes accelerate peft trl qwen-vl-utils paddleocr pdf2image
!apt-get -qq install -y poppler-utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295

In [ ]:
from google.colab import files
uploaded = files.upload()  # click the button and pick your PDF

Saving 7-12-sample-of-Maharashtra.pdf to 7-12-sample-of-Maharashtra.pdf


In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

# 4-bit NF4 Quantization (keeps VRAM around ~6-7 GB on your 15GB T4 GPU)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

# Resolution bounds to prevent VRAM spikes
processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28
)

print("⏳ Downloading and loading Qwen2.5-VL-7B in 4-bit (takes ~2 minutes)...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=compute_dtype,
    quantization_config=bnb_config
)

print("✅ Qwen2.5-VL-7B loaded successfully into Colab GPU!")

⏳ Downloading and loading Qwen2.5-VL-7B in 4-bit (takes ~2 minutes)...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

✅ Qwen2.5-VL-7B loaded successfully into Colab GPU!


In [ ]:
!pip install -q peft trl

cell 4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted!")

Mounted at /content/drive
✅ Drive mounted!


In [ ]:
model.save_pretrained("/content/drive/MyDrive/marathi-ocr-model/lora_adapter")
processor.save_pretrained("/content/drive/MyDrive/marathi-ocr-model/lora_adapter")
print("✅ Model saved!")

✅ Model saved!


In [ ]:
!pip install -q transformers bitsandbytes accelerate peft qwen-vl-utils pdf2image
!apt-get -qq install -y poppler-utils

from google.colab import drive
drive.mount('/content/drive')
print("✅ Ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 70.5 MB/s eta 0:00:00
Selecting previously unselected package poppler-utils.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.9_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.9) ...
Setting up poppler-utils (24.02.0-1ubuntu9.9) ...
Processing triggers for man-db (2.12.0-4build2) ...
Mounted at /content/drive
✅ Ready!


loading saved (LoRA) weights
|

In [ ]:
from google.colab import files
from pdf2image import convert_from_path

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

# Convert PDF pages into images at 300 DPI
images = convert_from_path(pdf_filename, dpi=300)
print(f"✅ Document loaded! Total pages: {len(images)}")

Saving 7-12-sample-of-Maharashtra.pdf to 7-12-sample-of-Maharashtra (1).pdf
✅ Document loaded! Total pages: 2


In [ ]:
import json, re
from PIL import ImageEnhance
from qwen_vl_utils import process_vision_info

def extract_universal_712(pdf_images):
    """
    Universal 7/12 Extractor for ANY Maharashtra district/layout.
    """
    # 1. Enhance contrast (removes watermarks, darkens faint handwriting)
    img_page1 = pdf_images[0].convert("RGB")
    enhanced_img = ImageEnhance.Contrast(img_page1).enhance(1.5)

    # 2. Universal Prompt (Pattern-based, no district bias)
    UNIVERSAL_PROMPT = """You are an official Maharashtra Land Records (भूमी अभिलेख) data extraction system.
Analyze this 7/12 (सातबारा उतारा) document and output ONLY a valid JSON object.

EXTRACTION PATTERNS & ANCHORS:
1. ADMINISTRATIVE HEADER:
   - "district": Text immediately following "जिल्हा :-" or "जिल्हा:"
   - "taluka": Text immediately following "तालुका :-" or "तालुका:"
   - "village": Text immediately following "गाव :-" or "गाव:" (Exclude the 6-digit code in brackets. Never output "गाव नमुना सात").
   - "village_code": The 6-digit number in brackets next to the village name. (null if absent).

2. LAND & TENURE:
   - "survey_number": The large number under "गट क्रमांक व उपविभाग" or "सर्व्हे नंबर".
   - "khata_number": The account number under "खाते क्रमांक" or "खाते क्र.".
   - "occupancy_class": Exact text under "भूधारणा पद्धती" (e.g. भोगवटदार वर्ग - १ or भोगवटदार वर्ग - २).

3. OWNERSHIP:
   - "owners": List of all human names listed under the "भोगवटदाराचे नाव" column. (Clean names, ignore lines like 'सामाईक क्षेत्र').

4. AREA MEASUREMENTS:
   - "area": {
       "unit": The unit specified (usually "हे.आर.चौ.मी" or "एकर-गुंठा"),
       "total": Number in "एकूण क्षेत्र (अ+ब)",
       "cultivable": Number in "एकूण ला.यो. क्षेत्र" or "लागवडी योग्य",
       "uncultivable": Number in "एकूण पो.ख.क्षेत्र" or "पोटखराब"
     }

5. LEGAL & FINANCIAL:
   - "assessment": Value under "आकारणी".
   - "encumbrances": Any bank loans, mortgages, or liens listed under "इतर अधिकार". If none or empty, return null.
   - "last_mutation_number": Approved mutation number mentioned under "शेवटचा फेरफार क्रमांक" or "प्रलंबित फेरफार".

RULES:
- Keep all text in original Marathi Devanagari script.
- Read Marathi numerals (०,१,२,३,४,५,६,७,८,९) accurately.
- If a section does not exist on this specific document, return null. DO NOT invent fake data.
- Output RAW JSON ONLY."""

    messages = [
        {"role": "system", "content": UNIVERSAL_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": enhanced_img},
                {"type": "text", "text": "Extract all fields from this Maharashtra 7/12 document according to the extraction rules."}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    vision_imgs, _ = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=vision_imgs,
        return_tensors="pt",
        padding=True,
        truncation=False
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=768,
            temperature=0.01
        )

    raw_output = processor.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    clean_text = raw_output.replace("```json", "").replace("```", "").strip()

    try:
        data = json.loads(clean_text)
    except Exception:
        match = re.search(r'\{.*\}', clean_text, re.DOTALL)
        data = json.loads(match.group(0)) if match else {"error": "JSON parse error", "raw": raw_output}

    return data

# Run on your uploaded document
result = extract_universal_712(images)
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "district": "पुणे",
  "taluka": "माळक",
  "village": "भोगवटदार",
  "village_code": "196023",
  "survey_number": "93",
  "khata_number": "1938",
  "occupancy_class": "भोगवटदार वर्ग - १",
  "owners": [
    "मोहनीश रमेश चावला"
  ],
  "area": {
    "unit": "हे.आर.चौ.मी",
    "total": "0.3960",
    "cultivable": "0.39",
    "uncultivable": "0.0060"
  },
  "assessment": "0.9950",
  "encumbrances": null,
  "last_mutation_number": "79/09/2019"
}


testing on some messy handwriting


In [1]:
from google.colab import files
uploaded = files.upload()  # click the button and pick your PDF

Saving Samples-from-the-dataset-of-handwritten-Devanagari-images.png to Samples-from-the-dataset-of-handwritten-Devanagari-images.png


In [4]:
!pip install -q transformers bitsandbytes accelerate qwen-vl-utils
print("✅ Libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 72.5 MB/s eta 0:00:00
✅ Libraries installed!


In [6]:
import torch
from PIL import Image
from google.colab import files
from qwen_vl_utils import process_vision_info
# Additions to fix NameError for 'processor' and 'model'
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

if 'processor' not in locals() or 'model' not in locals(): # Check if processor and model are already defined
    model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True
    )
    processor = AutoProcessor.from_pretrained(
        model_id,
        min_pixels=256 * 28 * 28,
        max_pixels=1280 * 28 * 28
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=compute_dtype,
        quantization_config=bnb_config
    )
    print("✅ Processor and Model re-loaded for current session.")

# 1. Upload your messy handwriting image (PNG or JPG)
print("Please upload your messy handwriting image...")
uploaded = files.upload()
img_filename = list(uploaded.keys())[0]

# 2. Open the image directly with PIL
raw_img = Image.open(img_filename).convert("RGB")
print("✅ Image loaded!")

# 3. Simple OCR Test Prompt
SYSTEM_PROMPT = """You are an expert Devanagari OCR system.
This image contains a grid of messy handwritten words.
Please transcribe all the words you can read, row by row, from left to right."""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": raw_img},
            {"type": "text", "text": "Read the handwritten words in this image."}
        ]
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
vision_imgs, _ = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=vision_imgs,
    return_tensors="pt",
    padding=True,
    truncation=False
).to(model.device)

print("⏳ Testing 7B Model on Messy Handwriting...")
with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.1)

raw_output = processor.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n📄 OCR Result:")
print(raw_output)

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

✅ Processor and Model re-loaded for current session.
Please upload your messy handwriting image...


Saving Samples-from-the-dataset-of-handwritten-Devanagari-images.png to Samples-from-the-dataset-of-handwritten-Devanagari-images (2).png
✅ Image loaded!
⏳ Testing 7B Model on Messy Handwriting...

📄 OCR Result:
सिक्कू
231670
1414
1/2
बालक
21409
कोलर
65158
कार
1290
तीन
(29)
में
745
हरिश्चन्द्र
1805
में
132
फर्मेंट
68
कोपुलेशन
(अवश्य)
0.181 - 31.2


In [7]:
SYSTEM_PROMPT = """You are a strict Hindi/Marathi handwriting expert.
You are looking at an image that contains a 6x4 grid of isolated, handwritten words.
CRITICAL RULES:
1. THERE ARE NO NUMBERS IN THIS IMAGE. Do not output any digits.
2. Read exactly what is written. For example, the first row contains words like 'सिकंदर' and 'मकान'.
3. Output the text row by row. If you cannot read a word, output [UNREADABLE]. Do not guess random words."""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": raw_img},
            {"type": "text", "text": "Transcribe the 6 rows of words."}
        ]
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
vision_imgs, _ = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=vision_imgs,
    return_tensors="pt",
    padding=True,
    truncation=False
).to(model.device)

with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.01)

raw_output = processor.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n📄 Improved OCR Result:")
print(raw_output)


📄 Improved OCR Result:
सिकंदर
बैली
तेज
कोप
कामगार
आवश्यक
